# Non-Circular Gears: Conjugate Gear Design

Classical gears are circular because a circular profile transmits rotation at a **constant speed ratio**. However, certain mechanical applications require **variable speed ratios** — a cam-like input producing a periodic output variation. **Non-circular gears** achieve this by replacing the circular profile with a closed curve.

## The conjugate gear problem

Given a driver gear with radial profile $r(\theta)$ (radius as a function of angle), find the **conjugate (driven) gear** $\rho(\phi)$ such that when the two gears mesh, they maintain **contact at all times** with teeth that roll without slipping.

## Rolling constraint and center distance

If the two gears are separated by a center distance $L$, the rolling constraint requires that at each contact angle:
$$
r(\theta) + \rho(\phi) = L.
$$
The no-slip condition (equal arc lengths) gives the **angular velocity relation**:
$$
\dot{\phi} = \frac{r(\theta)}{L - r(\theta)}\,\dot{\theta},
$$
so the conjugate angle satisfies:
$$
\phi(\theta) = \int_0^\theta \frac{r(\alpha)}{L - r(\alpha)}\,d\alpha.
$$
For one full revolution of the driver ($\theta \in [0, 2\pi/K]$ for a $K$-lobe gear), the conjugate gear completes the same arc: $\phi(2\pi/K) = 2\pi/K$.

## Finding the center distance $L$

$L$ is determined by the period-matching constraint:
$$
\int_0^{2\pi} \frac{r(\alpha)}{L - r(\alpha)}\,d\alpha = 2\pi.
$$
This nonlinear equation is solved by bisection: $L$ must satisfy $L > r_{\max}$, and the integral is monotonically decreasing in $L$.

## Tooth profile

The smooth radial profile is decorated with **involute teeth** in a real gear. Here we focus on the pitch curves (smooth profiles) that determine the velocity ratio, abstracting away tooth geometry.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

plt.rcParams['figure.dpi'] = 120

## Computing the conjugate gear

Given the driver radial profile $r(\theta)$ sampled on $n$ equally spaced angles, we:
1. Find $L$ by bisection (period-matching constraint).
2. Compute the conjugate angles $\phi(\theta)$ by cumulative integration.
3. Evaluate the conjugate radius $\rho(\phi) = L - r(\theta(\phi))$ by interpolation.

In [ ]:
def compute_conjugate_gear(r, K=1, n_iter=60):
    """
    Compute the conjugate gear given radial profile r (length n, one period = 2pi/K).
    Returns (rho, L, phi_map) where rho is the conjugate radial profile.
    """
    n = len(r)
    dphi = 2 * np.pi / (K * n)  # angle step

    def eval_integral(L):
        return np.sum(r / (L - r)) * dphi - 2 * np.pi / K

    # Bisection to find L
    L1, L2 = r.max() * 1.001, r.max() * 6.0
    for _ in range(n_iter):
        L = (L1 + L2) / 2
        if eval_integral(L) < 0:
            L2 = L
        else:
            L1 = L
    L = (L1 + L2) / 2

    # Conjugate angle mapping
    phi = np.concatenate([[0], np.cumsum(r / (L - r)) * dphi])
    phi = phi / phi[-1] * 2 * np.pi  # normalize
    theta_vals = np.linspace(0, 2 * np.pi, n + 1)

    # Backward mapping: theta(phi) → rho(phi)
    phi_uniform = np.linspace(0, 2 * np.pi, n + 1)
    r_full = np.append(r, r[0])  # periodic
    theta_at_phi = np.interp(phi_uniform, phi, theta_vals)
    rho = L - np.interp(theta_at_phi, theta_vals, r_full)

    # Reverse rotation direction for conjugate
    phi_cw = 2 * np.pi - phi_uniform[::-1]

    return rho[:-1], L, phi


def polar_to_cart(r, offset_x=0.0):
    n = len(r)
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    x = r * np.cos(theta) + offset_x
    y = r * np.sin(theta)
    return x, y


print('Conjugate gear solver ready.')

## Gallery of non-circular gear pairs

We design several driver profiles and compute their conjugate gears. Profiles include elliptical, limaçon, and multi-lobe sinusoidal shapes.

In [ ]:
n = 512
theta = np.linspace(0, 2 * np.pi, n, endpoint=False)

# Define driver profiles
profiles = {
    'ellipse':      1.0 + 0.35 * np.cos(theta),
    '3-lobe':       1.0 + 0.25 * np.cos(3 * theta),
    'limaçon':      1.0 + 0.4 * np.cos(theta) + 0.15 * np.cos(2 * theta),
    '5-lobe':       1.0 + 0.15 * np.cos(5 * theta),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
cols = ['royalblue', 'tomato', 'seagreen', 'darkorange']

for ax, (name, r_driver), col in zip(axes.ravel(), profiles.items(), cols):
    rho, L, _ = compute_conjugate_gear(r_driver)

    x1, y1 = polar_to_cart(r_driver, offset_x=0)
    x2, y2 = polar_to_cart(rho, offset_x=L)

    ax.fill(x1, y1, alpha=0.3, color=col)
    ax.plot(np.append(x1, x1[0]), np.append(y1, y1[0]), lw=2, color=col, label='driver')
    ax.fill(x2, y2, alpha=0.3, color='gray')
    ax.plot(np.append(x2, x2[0]), np.append(y2, y2[0]), lw=2, color='gray', label='conjugate')
    ax.plot(0, 0, 'k.', ms=8); ax.plot(L, 0, 'k.', ms=8)
    ax.set_aspect('equal'); ax.grid(alpha=0.2)
    ax.set_title(f'{name} (L={L:.3f})', fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('Non-circular gear pairs: driver (colored) and conjugate (gray)', y=1.02)
plt.tight_layout()
plt.show()

## Velocity ratio

The instantaneous speed ratio $\omega_2 / \omega_1 = r(\theta) / (L - r(\theta)) = r(\theta) / \rho(\phi)$ varies with angle. For the elliptic driver it oscillates between $r_{\min}/r_{\max}$ and $r_{\max}/r_{\min}$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
theta_deg = np.degrees(theta)

for (name, r_driver), col in zip(profiles.items(), cols):
    rho, L, _ = compute_conjugate_gear(r_driver)
    speed_ratio = r_driver / (L - r_driver)
    ax.plot(theta_deg, speed_ratio, lw=2, color=col, label=name)

ax.axhline(1.0, color='k', lw=1, ls='--', label='constant ratio (circular)')
ax.set_xlabel(r'driver angle $\theta$ (degrees)')
ax.set_ylabel(r'speed ratio $\omega_2/\omega_1$')
ax.set_title('Instantaneous speed ratio of non-circular gear pairs')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Gear meshing animation (static frame)

We show the two gears at a specific contact angle $\theta_0$: the driver is at angle $\theta_0$ and the conjugate gear at the corresponding angle $\phi(\theta_0)$, with the contact point on the line between centers.

In [ ]:
r_driver = profiles['ellipse']
rho, L, phi_map = compute_conjugate_gear(r_driver)

def plot_meshing(theta0_deg=0):
    theta0 = np.radians(theta0_deg)
    idx = int(theta0_deg / 360 * n) % n
    phi0 = phi_map[idx]

    # Rotate driver by theta0, conjugate by phi0 (in opposite direction)
    theta_r = theta + theta0
    x1 = r_driver * np.cos(theta_r)
    y1 = r_driver * np.sin(theta_r)

    phi_r = theta - phi0  # reversed
    x2 = rho * np.cos(phi_r) + L
    y2 = rho * np.sin(phi_r)

    # Contact point
    r_contact = r_driver[idx]
    cx = r_contact * np.cos(theta0)
    cy = r_contact * np.sin(theta0)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.fill(x1, y1, alpha=0.3, color='royalblue')
    ax.plot(np.append(x1, x1[0]), np.append(y1, y1[0]), lw=2, color='royalblue', label='driver')
    ax.fill(x2, y2, alpha=0.3, color='tomato')
    ax.plot(np.append(x2, x2[0]), np.append(y2, y2[0]), lw=2, color='tomato', label='conjugate')
    ax.plot([0, L], [0, 0], 'k--', lw=1, alpha=0.5)
    ax.plot(0, 0, 'k.', ms=10); ax.plot(L, 0, 'k.', ms=10)
    ax.plot(cx, cy, 'g*', ms=15, label='contact point')
    ax.set_aspect('equal'); ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
    ax.set_title(fr'Elliptic gear pair at $\theta={theta0_deg}°$')
    plt.tight_layout(); plt.show()

interact(plot_meshing,
         theta0_deg=IntSlider(value=0, min=0, max=355, step=5,
                              description='$\\theta$ (deg)'));

## Bibliographical resources

- Litvin, F. L. and Fuentes, A. (2004). *Gear Geometry and Applied Theory* (2nd ed.). Cambridge University Press.
- Dooner, D. B. (2012). *Kinematic Geometry of Gearing* (2nd ed.). Wiley.
- Chironis, N. P. and Sclater, N. (1996). *Mechanisms and Mechanical Devices Sourcebook* (2nd ed.). McGraw-Hill.
- Mundo, D. (2006). Geometric design of a planetary gear train with non-circular gears. *Mechanism and Machine Theory*, 41(4), 456–472.
- Bair, B. W. (2002). Computer aided design of non-standard elliptical gear drives. *Proceedings of the Institution of Mechanical Engineers, Part C*, 216(4), 473–483.